### 🧩 Step 1 — Load the Cleaned Dataset
We begin by loading the cleaned FPL dataset that already includes the `form` column.
This will be our starting point before we engineer the target column and finalize features.


In [66]:
import pandas as pd

# Load cleaned dataset (make sure the filename matches your actual file)
df = pd.read_csv("../data/cleaned/cleaned_merged_seasons_with_form.csv")

# Display basic info
df.info()
df.head()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 96169 entries, 0 to 96168
Data columns (total 38 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   season_x           96169 non-null  object 
 1   name               96169 non-null  object 
 2   position           96169 non-null  object 
 3   team_x             96169 non-null  object 
 4   assists            96169 non-null  int64  
 5   bonus              96169 non-null  int64  
 6   bps                96169 non-null  int64  
 7   clean_sheets       96169 non-null  int64  
 8   creativity         96169 non-null  float64
 9   element            96169 non-null  int64  
 10  fixture            96169 non-null  int64  
 11  goals_conceded     96169 non-null  int64  
 12  goals_scored       96169 non-null  int64  
 13  ict_index          96169 non-null  float64
 14  influence          96169 non-null  float64
 15  kickoff_time       96169 non-null  object 
 16  minutes            961

,season_x,name,position,team_x,assists,bonus,bps,clean_sheets,creativity,element,...,threat,total_points,transfers_balance,transfers_in,transfers_out,value,was_home,yellow_cards,GW,form
0,2020-21,Aaron Connolly,FWD,Brighton,0,0,-3,0,0.3,78,...,32.0,1,0,0,0,55,True,0,1,0.100000
1,2020-21,Aaron Connolly,FWD,Brighton,0,2,27,1,11.3,78,...,23.0,8,-1161,5332,6493,55,False,0,2,0.450000
2,2020-21,Aaron Connolly,FWD,Brighton,0,0,2,0,12.1,78,...,8.0,2,13526,26823,13297,55,True,0,3,0.366667
3,2020-21,Aaron Connolly,FWD,Brighton,0,0,7,0,0.3,78,...,4.0,2,-1311,10399,11710,55,False,0,4,0.325000
4,2020-21,Aaron Connolly,FWD,Brighton,1,0,13,0,10.3,78,...,2.0,4,-8992,5860,14852,55,False,0,5,0.400000


### 🧮 Step 2 — Delete duplicate game weeks



In [67]:
# Remove duplicate (player, GW) entries by keeping the one where the player actually played
df = df.sort_values(["name", "GW", "minutes"], ascending=[True, True, False])

# Drop duplicates — keep the one with highest minutes for each player/week
df = df.drop_duplicates(subset=["name", "GW"], keep="first").reset_index(drop=True)

### 🎯 Step 3 — Create the Target Column (`upcoming_total_points`)
We want to predict each player's total points for the *following* gameweek.  
So for each player, we shift the `total_points` column one step upward.  
The last gameweek for each player will have no "next week" value, so those rows are dropped.


In [68]:
df["upcoming_total_points"] = df.groupby("name")["total_points"].shift(-1)
df = df.dropna(subset=["upcoming_total_points"]).reset_index(drop=True)

df[["name", "GW", "total_points", "upcoming_total_points"]].head(10)



,name,GW,total_points,upcoming_total_points
0,Aaron Connolly,1,1,8.0
1,Aaron Connolly,2,8,2.0
2,Aaron Connolly,3,2,2.0
3,Aaron Connolly,4,2,4.0
4,Aaron Connolly,5,4,1.0
5,Aaron Connolly,6,1,0.0
6,Aaron Connolly,7,0,1.0
7,Aaron Connolly,8,1,0.0
8,Aaron Connolly,9,0,2.0
9,Aaron Connolly,10,2,2.0


### 🧹 Step 4 — Remove Irrelevant Columns
According to the project description, we exclude columns that depend on player popularity
or are identifiers that don’t contribute to predictive modeling.


In [69]:
drop_cols = [
    "transfers_in", "transfers_out", "transfers_balance",
    "element", "season_x"
]

df = df.drop(columns=[c for c in drop_cols if c in df.columns], errors="ignore")


### ⚙️ Step 5 — Select Match-Related and Player-Related Features
We now focus on columns describing performance or player characteristics.  
These will serve as our input features for the regression model.


In [70]:
feature_cols = [
    "total_points",  
    "minutes", "goals_scored", "assists", "bonus", "bps",
    "clean_sheets", "creativity", "influence", "threat",
    "ict_index", "form", "value", "was_home", "yellow_cards", "red_cards",
    "position", "team_x"
]

target_col = "upcoming_total_points"

keep_cols = ["name", "GW"] + feature_cols + [target_col]
df = df[[c for c in keep_cols if c in df.columns]]


### 🔠 Step 6 — Encode Categorical Variables
We convert text features like `team` and `position` into numeric dummy variables.
`drop_first=True` avoids multicollinearity by omitting one category from each.


In [71]:
df["was_home"] = df["was_home"].astype(int)

df = pd.get_dummies(df, columns=["position", "team_x"], drop_first=True)
print("Categorical features encoded successfully.")


Categorical features encoded successfully.


### 💾 Step 7 — Save Final Prepared Dataset
The dataset is now fully prepared for model training.
We save it as `final_prepared_dataset.csv`, ready for splitting into train/test sets.


In [72]:
output_path = "../data/cleaned/final_prepared_dataset.csv"
df.to_csv(output_path, index=False)

print(f"✅ Final dataset saved as '{output_path}'")
print("Shape:", df.shape)


✅ Final dataset saved as '../data/cleaned/final_prepared_dataset.csv'
Shape: (41962, 46)


### 📊  Sanity Check
Verify that there are no missing values and inspect summary statistics before splitting.


In [73]:
print(df.isna().sum().sum(), "missing values in the final dataset.")
df.describe().T.head(15)


0 missing values in the final dataset.


,count,mean,std,min,25%,50%,75%,max
GW,41962.0,19.998665,10.613773,1.00,11.0,21.000,29.00,37.0
total_points,41962.0,1.686764,2.727116,-4.00,0.0,1.000,2.00,29.0
minutes,41962.0,40.819980,42.393146,0.00,0.0,16.000,90.00,90.0
goals_scored,41962.0,0.055860,0.257351,0.00,0.0,0.000,0.00,4.0
assists,41962.0,0.049569,0.233556,0.00,0.0,0.000,0.00,3.0
bonus,41962.0,0.136290,0.548272,0.00,0.0,0.000,0.00,3.0
bps,41962.0,7.483247,10.468864,-18.00,0.0,2.000,13.00,104.0
clean_sheets,41962.0,0.132191,0.338702,0.00,0.0,0.000,0.00,1.0
creativity,41962.0,5.973884,11.746943,0.00,0.0,0.000,6.20,170.9
influence,41962.0,8.965135,13.845966,0.00,0.0,0.200,14.40,163.6


In [74]:
df.shape
df.columns


Index(['name', 'GW', 'total_points', 'minutes', 'goals_scored', 'assists',
       'bonus', 'bps', 'clean_sheets', 'creativity', 'influence', 'threat',
       'ict_index', 'form', 'value', 'was_home', 'yellow_cards', 'red_cards',
       'upcoming_total_points', 'position_FWD', 'position_GK', 'position_MID',
       'team_x_Aston Villa', 'team_x_Bournemouth', 'team_x_Brentford',
       'team_x_Brighton', 'team_x_Burnley', 'team_x_Chelsea',
       'team_x_Crystal Palace', 'team_x_Everton', 'team_x_Fulham',
       'team_x_Leeds', 'team_x_Leicester', 'team_x_Liverpool',
       'team_x_Man City', 'team_x_Man Utd', 'team_x_Newcastle',
       'team_x_Norwich', 'team_x_Nott'm Forest', 'team_x_Sheffield Utd',
       'team_x_Southampton', 'team_x_Spurs', 'team_x_Watford',
       'team_x_West Brom', 'team_x_West Ham', 'team_x_Wolves'],
      dtype='object')

In [75]:
sample = df[df["name"] == "Erling Haaland"][["GW", "total_points", "upcoming_total_points"]].head(10)
print(sample)


       GW  total_points  upcoming_total_points
11658   1            13                    5.0
11659   2             5                    6.0
11660   3             6                   17.0
11661   4            17                   17.0
11662   5            17                    9.0
11663   6             9                    6.0
11664   8             6                   23.0
11665   9            23                    6.0
11666  10             6                    2.0
11667  11             2                   13.0


In [76]:
dupes = df.duplicated(subset=["name", "GW"]).sum()
print("Duplicate (name, GW) pairs:", dupes)


Duplicate (name, GW) pairs: 0


In [79]:
# Shift total_points up by one within each player to represent the next GW's actual points
shifted = df.groupby("name")["total_points"].shift(-1)

# Compare current upcoming_total_points vs next GW's actual total_points
mismatch_mask = df["upcoming_total_points"] != shifted

# Filter to mismatches (excluding NaNs for final GWs)
mismatches = df[mismatch_mask & df["upcoming_total_points"].notna()]

print(f"✅ Alignment check complete — mismatches found: {len(mismatches)}")

# If any mismatches exist, show a few examples
if len(mismatches) > 0:
    display(df.loc[mismatches.index, ["name", "GW", "total_points", "upcoming_total_points"]].head(10))

✅ Alignment check complete — mismatches found: 1320


,name,GW,total_points,upcoming_total_points
36,Aaron Connolly,37,0,1.0
73,Aaron Cresswell,37,0,2.0
108,Aaron Hickey,37,5,6.0
145,Aaron Lennon,37,2,2.0
182,Aaron Mooy,37,3,2.0
219,Aaron Ramsdale,37,3,9.0
256,Aaron Ramsey,37,2,10.0
293,Aaron Wan-Bissaka,37,2,3.0
330,Abdoulaye Doucouré,37,2,1.0
364,Aboubakar Kamara,37,0,0.0
